# Taller Taxis NYC — EMR + PySpark

Viajes de los taxis amarillos de Nueva York en 2018: **102.871.387 viajes** en 12 archivos Parquet
(1,46 GB), leídos desde S3 por un clúster EMR de 1 primario + 2 core `m5.large`.

**Kernel: PySpark.** La sesión `spark` la crea el kernel, en el clúster.

| Decisión | Valor |
|---|---|
| Dato erróneo | recogida fuera de 2018 · duración ≤ 0 · duración mayor al tope (elegido con los percentiles) · zonas 264 y 265 |
| Qué se guarda (paso 12) | sólo 2018, con todas sus duraciones |

In [ ]:
# Parámetros del taller
BUCKET = "taxis-nyc-936980625651"
BASE = f"s3://{BUCKET}"
TOPE_MIN = None          # duración máxima válida en minutos: la marcan los percentiles (referencia de Athena)
SIN_ZONA = [264, 265]    # 264 = Unknown · 265 = Outside of NYC: ninguna es un lugar

from pyspark import StorageLevel
from pyspark.sql import functions as F

# Las fechas del TLC son hora local de Nueva York sin zona horaria. Con la sesión en UTC,
# year(), month() y la resta trabajan sobre la hora tal como está escrita.
spark.conf.set("spark.sql.session.timeZone", "UTC")
assert TOPE_MIN is not None, "Completar TOPE_MIN con el tope elegido en la referencia de Athena"

## Paso 2 — Viajes y zonas, en dos DataFrames

Se leen **sólo las 4 columnas** que usa el taller, con un esquema explícito: tienen el mismo tipo en
los 12 meses. Leer la carpeta con su esquema completo puede fallar, porque `passenger_count` y
`RatecodeID` pasan de INT64 a DOUBLE en julio.

In [ ]:
ESQUEMA_VIAJES = ("tpep_pickup_datetime timestamp, tpep_dropoff_datetime timestamp, "
                  "PULocationID long, DOLocationID long")
ESQUEMA_ZONAS = "LocationID int, Borough string, Zone string, service_zone string"

viajes = spark.read.schema(ESQUEMA_VIAJES).parquet(f"{BASE}/taxis/viajes/")
zonas = spark.read.csv(f"{BASE}/taxis/zonas/", header=True, schema=ESQUEMA_ZONAS)

print(f"viajes: {viajes.count():,}")   # control: 102.871.387
print(f"zonas:  {zonas.count():,}")    # control: 265
viajes.printSchema()
zonas.show(5, truncate=False)

## Paso 3 — Lugar de recogida (`PU_ZONE`), hora de recogida, hora de dejada y `DOLocationID`

Zonas tiene 265 filas: `broadcast` la copia a cada nodo y los 102 millones de viajes no se mueven
por la red.

In [ ]:
zonas_pu = zonas.select(F.col("LocationID").alias("PULocationID"),
                        F.col("Zone").alias("PU_ZONE"))

paso3 = (viajes.join(F.broadcast(zonas_pu), on="PULocationID")
               .select("PULocationID", "PU_ZONE",
                       F.col("tpep_pickup_datetime").alias("hora_recogida"),
                       F.col("tpep_dropoff_datetime").alias("hora_dejada"),
                       "DOLocationID"))

paso3.select("PU_ZONE", "hora_recogida", "hora_dejada", "DOLocationID").show(10, truncate=False)

## Paso 4 — Segundo cruce: el nombre de la zona de destino

Los IDs quedan guardados para agrupar en los pasos 7 y 8: hay nombres de zona repetidos
("Corona" es la 56 y la 57), y agrupar por nombre sumaría zonas distintas.

In [ ]:
zonas_do = zonas.select(F.col("LocationID").alias("DOLocationID"),
                        F.col("Zone").alias("DO_ZONE"))

paso4 = (paso3.join(F.broadcast(zonas_do), on="DOLocationID")
              .select("PULocationID", "DOLocationID", "PU_ZONE", "DO_ZONE",
                      F.col("hora_recogida").alias("PU_datetime"),
                      F.col("hora_dejada").alias("DO_datetime")))

paso4.select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime").show(10, truncate=False)

## Paso 5 — `trip_duration`, en minutos

Segundos Unix de la bajada menos los de la recogida, dividido 60. El resultado se guarda en
memoria (y en disco lo que no entre): todo lo que sigue sale de acá, y así no se relee S3 ni se
repiten los joins en cada consulta.

In [ ]:
viajes_dur = paso4.withColumn(
    "trip_duration",
    (F.unix_timestamp("DO_datetime") - F.unix_timestamp("PU_datetime")) / 60)

viajes_dur.persist(StorageLevel.MEMORY_AND_DISK)
print(f"después de los dos joins: {viajes_dur.count():,}")   # control: igual que viajes

viajes_dur.select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime",
                  F.round("trip_duration", 2).alias("trip_duration")).show(10, truncate=False)

## Antes de responder: qué tan sucios están los datos

Válido = recogida en 2018 y duración entre 0 y el tope. Los percentiles son los que justifican el
tope elegido.

In [ ]:
es_2018 = F.year("PU_datetime") == 2018
dur = F.col("trip_duration")
valido = es_2018 & (dur > 0) & (dur <= TOPE_MIN)

viajes_dur.select(
    F.count("*").alias("total"),
    F.sum((~es_2018).cast("int")).alias("fuera_de_2018"),
    F.sum((es_2018 & (dur <= 0)).cast("int")).alias("duracion_no_positiva"),
    F.sum((es_2018 & (dur > TOPE_MIN)).cast("int")).alias("sobre_el_tope"),
    F.sum(valido.cast("int")).alias("validos"),
).show(truncate=False)

percentiles = viajes_dur.filter(es_2018 & (dur > 0)).approxQuantile(
    "trip_duration", [0.5, 0.9, 0.99, 0.999, 0.9999], 0.0001)
print("percentiles de duración (min) p50, p90, p99, p99.9, p99.99:", [round(p, 1) for p in percentiles])

## Paso 6 — El viaje que más duró

Dos respuestas: el más largo **tal como viene** —que muestra el dato roto— y el más largo
**entre los válidos**.

In [ ]:
columnas = ["PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime",
            F.round("trip_duration", 1).alias("trip_duration"),
            F.round(F.col("trip_duration") / 60 / 24, 1).alias("dias")]

print("Tal como viene:")
viajes_dur.orderBy(F.desc("trip_duration")).select(*columnas).show(1, truncate=False)

print(f"Entre los válidos (2018, entre 0 y {TOPE_MIN:g} min):")
viajes_dur.filter(valido).orderBy(F.desc("trip_duration")).select(*columnas).show(1, truncate=False)

## Paso 7 — El destino más frecuente, sin datos erróneos ni las zonas 264 y 265

In [ ]:
destinos = (viajes_dur
            .filter(valido & ~F.col("DOLocationID").isin(SIN_ZONA))
            .groupBy("DOLocationID", "DO_ZONE")
            .count()
            .orderBy(F.desc("count")))
destinos.show(5, truncate=False)

## Paso 8 — El viaje más frecuente (origen-destino): promedio y desviación estándar

`stddev` es la desviación estándar **muestral**. Los valores absurdos la inflan: por eso va con el
filtro de validez.

In [ ]:
pares = (viajes_dur
         .filter(valido
                 & ~F.col("PULocationID").isin(SIN_ZONA)
                 & ~F.col("DOLocationID").isin(SIN_ZONA))
         .groupBy("PULocationID", "PU_ZONE", "DOLocationID", "DO_ZONE")
         .agg(F.count("*").alias("viajes"),
              F.round(F.avg("trip_duration"), 2).alias("promedio_min"),
              F.round(F.stddev("trip_duration"), 2).alias("desviacion_estandar_min"))
         .orderBy(F.desc("viajes")))
pares.show(5, truncate=False)

## Pasos 9 y 10 — Año y mes de la recogida, como enteros

In [ ]:
viajes_ym = (viajes_dur
             .withColumn("year", F.year("PU_datetime").cast("int"))
             .withColumn("month", F.month("PU_datetime").cast("int")))
viajes_ym.printSchema()

## Paso 11 — Meses de 2018, de más a menos viajes

Sólo 2018: si no, enero sumaría también los viajes de enero de 2001.

In [ ]:
del_2018 = viajes_ym.filter(F.col("year") == 2018)
meses = del_2018.groupBy("month").count().orderBy(F.desc("count"))
meses.show(12)

## Paso 12 — Parquet particionado en `s3://bucket/viajes/year=…/month=…`

Se guarda 2018 con todas sus duraciones: 12 particiones, un archivo por mes.

In [ ]:
salida = f"{BASE}/viajes/"
(del_2018
 .select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime", "trip_duration", "year", "month")
 .repartition("year", "month")   # un archivo por mes: buen tamaño para Athena
 .write.mode("overwrite")
 .partitionBy("year", "month")
 .parquet(salida))

guardado = spark.read.parquet(salida)
print(f"guardadas: {guardado.count():,}")
guardado.groupBy("year", "month").count().orderBy("month").show(12)

## Pasos 13 y 14

- **13** — Crawler de Glue sobre `s3://taxis-nyc-936980625651/viajes/` → tabla `viajes` en la base
  `taxis`, consultada desde Athena (`infra/04-crawler.sh`).
- **14** — Este notebook, convertido en `spark/taller_taxis.py` y corrido como EMR Step
  (`infra/03-correr-paso-14.sh`). Tiene que dar los mismos números.